<font color=red>**Предупреждение:**</font> в этой работе ты будешь дообучать модель генерировать положительные, отрицательные и даже токсичные отзывы. Здесь это учебный эксперимент, но сходные методы могут использоваться для [массовых атак на рейтинги (review bombing)](https://en.wikipedia.org/wiki/Review_bomb), ботоферм в соцсетях и других сомнительных целей. Решение о применении знаний остаётся за тобой; стоит подумать, ради чего ты изучаешь ML.

# Выравнивание поведения LLM через обучение с подкреплением по обратной связи от людей (RLHF)

_Основано на [оригинальном ноутбуке](https://github.com/antndlcrx/oxford-llms-workshop/blob/main/materials/seminars/day_3/8_LLMs%20alignment%20with%20RLHF.ipynb) Ильи Бойцова для Oxford LLMs Workshop._

> Неофициальный перевод [исходного задания](./homework.ipynb). Программный код и проверки сохранены.

# Настройка окружения

Оригинальный семинар Ильи Бойцова основан на этом [примере](https://huggingface.co/docs/trl/v0.1.1/en/sentiment_tuning). В то время библиотека `trl` предоставляла инфраструктуру для дообучения языковых моделей через **PPO**.

В последующие годы методы RLHF продолжали развиваться, и PPO убрали из актуальной версии `trl`. Тем не менее исторически он важен; рекомендуем разобраться с ним, чтобы понимать преимущества более новых методов.

Сам PPO вырос из более ранних методов обучения с подкреплением — подробнее см. литературу и курсы по RL.

Поскольку оригинальный семинар подготовлен в конце 2023 года, настроим среду, максимально близкую к тогдашней, с учётом изменений безопасности PyTorch.

Ниже несколько конфигураций, проверенных авторами **19 октября 2026 года**.

## Вариант 1. Локально

Установи Python 3.10.14, например с помощью [pyenv](https://github.com/pyenv/pyenv).

Создай виртуальное окружение:

```bash
python -m venv 2026_10_19_trl_0_7_5_ve
```

Активируй его.

Установи `pip-tools`:

```bash
pip install pip-tools
```

Создай `requirements.in` со следующим содержимым:

```text
trl==0.7.5
peft==0.7.1
datasets
transformers==4.36.0
torch==2.1.1
numpy==1.23.5
accelerate==0.25.0
```

Сгенерируй `requirements.txt`:

```bash
pip-compile -v requirements.in
```

Установи зависимости:

```bash
pip install -r requirements.txt
```

Наконец установи JupyterLab:

```bash
pip install jupyter
```

## Вариант 2. Google Colab

Подключись к среде с GPU T4.

Открой меню **Change runtime type** («Изменить тип среды выполнения») и выбери **Runtime version 2025.10**. Дождись пересоздания окружения. В этой среде Python 3.12.12.

Установи пакеты:

```python
!pip install torch==2.2.1 trl==0.8.0 transformers==4.38.2 accelerate==0.28.0 peft==0.9.0 numpy==1.26.2 huggingface-hub==0.25.2
```

Затем удали `diffusers`:

```python
!pip3 uninstall -y diffusers
```

Если появится сообщение о необходимости перезапуска runtime, перезапусти его.

После этого следующий код должен успешно выполняться:

```python
import trl
```

Теперь можно запускать код для `trl==0.8.0`, включая `PPOTrainer`, `PPOConfig` и `AutoModelForCausalLMWithValueHead`.

## Введение

На этом занятии дообучим языковую модель методом **обучения с подкреплением** так, чтобы она генерировала хорошие (или плохие) отзывы.

Воспользуемся библиотекой [Transformer Reinforcement Learning (TRL)](https://huggingface.co/docs/trl). Она реализует компоненты RLHF, включая обучение модели вознаграждения и PPO.

![img](https://huggingface.co/datasets/trl-internal-testing/example-images/resolve/main/images/TRL-readme.png)

### Практикум: настраиваем модель на положительные отзывы о фильмах

Чтобы понять TRL, настроим GPT-2 на корпусе IMDB, чтобы он генерировал положительные (или отрицательные) рецензии. **Выбирай сам, какой тип отзывов нужен.**

Но сначала посмотрим на базовую модель — GPT-2, ранее дообученную генерировать произвольные отзывы о фильмах.

In [ ]:
import torch
import transformers
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
main_tokenizer = transformers.AutoTokenizer.from_pretrained("lvwerra/gpt2-imdb")
main_model = transformers.AutoModelForCausalLM.from_pretrained("lvwerra/gpt2-imdb", device_map=device)

In [ ]:
inputs = main_tokenizer("The movie", return_tensors='pt').to(device)
generated_ids = main_model.generate(**inputs, max_new_tokens=50, do_sample=True)
print("\nGenerated text:", main_tokenizer.decode(generated_ids.flatten().cpu().numpy().tolist()))

Если несколько раз запустить эту ячейку, модель с некоторой вероятностью выдаст положительный, отрицательный или нейтральный отзыв. Далее обучим её создавать больше отзывов выбранной тональности.

Аналогично InstructGPT, процесс состоит из двух этапов:

1. **Обучить модель вознаграждения (reward model)** назначать большие значения положительным (или отрицательным) отзывам.
2. Дообучить языковую модель **максимизировать это вознаграждение** через [Proximal Policy Optimization (PPO)](https://openai.com/research/openai-baselines-ppo).

## Этап 1. Обучение reward model (1 балл)

Сначала обучим похожую на BERT модель вознаграждения. Для имитации человеческих предпочтений создадим синтетические парные сравнения.

**Вопрос:** зачем нужна reward model? Нельзя ли просто использовать готовый классификатор тональности?

**Ответ:** можно, но он подходит лишь для отзывов о фильмах. В этом практикуме мы научимся применять RLHF к произвольным целям.

**Если цель — максимизировать тональность или иную фиксированную метку, а не предпочтения людей, обучай reward model как обычный классификатор** (см. неделю 5).

In [ ]:
# Сейчас дообучаем небольшую BERT-подобную модель. В основном задании попробуй другие модели.
reward_model = transformers.AutoModelForSequenceClassification.from_pretrained("distilbert-base-cased", device_map=device)
reward_tokenizer = transformers.AutoTokenizer.from_pretrained("distilbert-base-cased")

**Обрати внимание:** у reward model свой токенизатор, отличный от токенизатора основной языковой модели. Для RLHF они не обязаны совпадать.

In [ ]:
# Для обучения reward model нужен датасет (или генератор) пар принятых/отклонённых ответов.
# Каждый обучающий пример — словарь из четырёх ключей:
#  - input_ids_chosen, attention_mask_chosen = tokenizer("A sentence that human labeler likes more")
#  - input_ids_rejected, attention_mask_rejected = tokenizer("A sentence that human labeler likes less")

import torch
import datasets

class IMDBPairwiseDataset(torch.utils.data.Dataset):
    """ A dataset of all possible pairs of chosen and texts in TRT reward training format """
    def __init__(self, imdb, tokenizer, accepted_label: int):
        super().__init__()
        self.tokenizer = tokenizer
        self.chosen_texts = [row['text'] for row in imdb if row['label'] == accepted_label]
        self.rejected_texts = [row['text'] for row in imdb if row['label'] != accepted_label]
        assert self.chosen_texts, f"no texts with label {accepted_label}"
        print(f"Found {len(self.chosen_texts)} chosen and {len(self.rejected_texts)} rejected texts, {len(self)} pairs")

    def __len__(self):
        return len(self.chosen_texts) * len(self.rejected_texts)  # all pairs

    def __getitem__(self, index: int):
        chosen = self.tokenizer(self.chosen_texts[index // len(self.chosen_texts)], truncation=True)
        rejected = self.tokenizer(self.rejected_texts[index % len(self.chosen_texts)], truncation=True)
        return dict(input_ids_chosen=chosen['input_ids'], attention_mask_chosen=chosen['attention_mask'],
                    input_ids_rejected=rejected['input_ids'], attention_mask_rejected=rejected['attention_mask'])

In [ ]:
TARGET_LABEL = <YOUR CHOICE - either 0 or 1>   # and make sure it works by reviewing the sample printed below
imdb = datasets.load_dataset("imdb", split='train')
reward_data = IMDBPairwiseDataset(imdb, reward_tokenizer, accepted_label=TARGET_LABEL)

sample = reward_data[31337]
print('CHOSEN:', reward_tokenizer.decode(sample['input_ids_chosen']))
print('REJECTED:', reward_tokenizer.decode(sample['input_ids_rejected']))

Будем использовать \`trl.RewardTrainer\` — специальный вариант знакомого \`transformers.Trainer\`. Он принимает те же типы параметров обучения (batch size, gradient checkpointing и др.), но обучает модель на **попарной функции вознаграждения** из [статьи InstructGPT](https://arxiv.org/pdf/2203.02155.pdf):

![Формула функции потерь](https://i.imgur.com/2JzNAPs.png)

Сама модель не оценивает пару одним вызовом: выбранный ($y_w$) и отвергнутый ($y_l$) образцы обрабатываются отдельно.

Чтобы минимизировать loss, reward model должна присваивать **более высокий балл принятому ответу**, чем отвергнутому.

Формула также учитывает контекст $x$, полезный для seq2seq. Для наших отзывов о фильмах $x$ пуст.

In [ ]:
import trl

training_args = trl.RewardConfig(  # like transformers.TrainingArguments
    output_dir="reward_model",
    per_device_train_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=1.41e-5,
    max_steps=1_000,              # note: training may need more than 1k steps
    logging_steps=50,
    gradient_checkpointing=True,  # reduce memory usage but train ~30% slower
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True                     # disable this on CPU or on very old GPUs
    # можно добавить другие полезные гиперпараметры из недель 5–7
)

trainer = trl.RewardTrainer(
    model=reward_model,
    args=training_args,
    tokenizer=reward_tokenizer,
    train_dataset=reward_data,
    peft_config=None,  # optionally, you may tune with LoRA, prompt-tuning, etc
)

trainer.train()

In [ ]:
reward_model.gradient_checkpointing_disable()
reward_model.eval()

### Проверка reward model (1 балл)

Посмотрим, насколько хорошо модель вознаграждения работает.

**Задание:** измерь, в какой доле пар (предпочитаемый и отвергнутый отзыв) она правильно определяет порядок. Измерь показатель **отдельно на train** (`imdb`) и на загружаемой ниже отдельной **test-выборке**.

In [ ]:

for sample_index in 45, 16000:
  print('TEXT:', imdb[sample_index]['text'])
  inputs = reward_tokenizer(
      imdb[sample_index]['text'], truncation=True, return_tensors='pt').to(device)
  with torch.no_grad():
    reward = reward_model(**inputs).logits[0, 0].item()
    print("REWARD:", reward)
  print('LABEL:', imdb[sample_index]['label'])
  print()

# reward model может выдавать другие абсолютные значения награды.
# Это нормально, если относительный порядок наград обычно правильный.

In [ ]:
imdb_test = datasets.load_dataset("imdb", split='test')

# <a whole lot of your code here, feel free to spit it as you see fit>

### Генерация с выбором по вознаграждению (1 балл)

Если всё сделано правильно, к этому моменту у тебя есть пригодная reward model. Прежде чем переходить к RL, попробуем улучшать результаты **без дополнительного обучения**.

Для этого можно использовать reward-guided inference: **сгенерировать N=16 вариантов, затем выбрать вариант с максимальным вознаграждением** согласно reward model.

**Задание:** самостоятельно докажи работоспособность подхода. Возьми как минимум **пять нейтральных промптов**, например `This movie is`, сгенерируй продолжения, оцени их reward model и покажи, какие тексты получили наибольшее вознаграждение.

Примечание: генерировать несколько примеров **параллельно батчем** быстрее, чем последовательно.

In [ ]:
inputs = main_tokenizer(["It was"] * 5, return_tensors='pt').to(device)
for candidate in main_model.generate(**inputs, max_new_tokens=50, do_sample=True):
  print("Sample:", main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()))

In [ ]:
# <YOUR CODE HERE> - feel free to organize it as you see fit

# Этап 2. Дообучение основной модели с RL (2 балла)

В этом практикуме будем оптимизировать GPT-2 так, чтобы он писал положительные отзывы IMDB, используя обученную выше reward model.

В отличие от supervised fine-tuning, RL позволяет модели **генерировать собственные тексты на каждом шаге обучения**. Затем вычисляется вознаграждение этих текстов, а веса обновляются так, чтобы повышать вероятность высоко оценённых результатов.

Каждая итерация RLHF состоит из трёх этапов: **Rollout**, **Evaluation**, **Update**.

<div style="text-align: center">
<img src='https://huggingface.co/datasets/trl-internal-testing/example-images/resolve/main/images/gpt2_bert_training.png' width='600'>

Реализация Update зависит от выбранного RL-алгоритма. Здесь используем [PPO](https://arxiv.org/abs/1707.06347), как в InstructGPT.

Прежде чем запускать эти этапы, создадим датасет «запросов» — в нашем случае это начала отзывов.

In [ ]:
# Этот код специфичен для IMDB: для других задач его придётся переписать.
imdb_for_rlhf = imdb.filter(lambda row: len(row['text']) > 200, batched=False)
imdb_for_rlhf = imdb_for_rlhf.remove_columns(['label'])
sample_length = trl.core.LengthSampler(2, 8)  # use the first 2-8 tokens as query

def select_query_and_tokenize(sample):
    query_ids = main_tokenizer.encode(sample["text"])[: sample_length()]
    sample["query"] = main_tokenizer.decode(query_ids)  # query is the only required column
    sample["input_ids"] = query_ids  # to avoid re-tokenizing later
    return sample  # we do not need the rest - it will be generated by the model

imdb_for_rlhf = imdb_for_rlhf.map(select_query_and_tokenize, batched=False)
imdb_for_rlhf.set_format(type="torch")

Подготовим reward model к оценке произвольных сгенерированных отзывов. Передаём отзывы как обычный текст, потому что основная модель и модель вознаграждения имеют **разные токенизаторы**.

In [ ]:
from typing import List
def compute_reward(texts: List[str]) -> torch.Tensor:
  inputs = reward_tokenizer(texts, truncation=True, padding=True, return_tensors='pt').to(device)
  with torch.no_grad():
    return reward_model(**inputs).logits[:, 0]

In [ ]:
compute_reward([imdb[45]['text'], imdb[16000]['text']])  # test on human-written reviews

Теперь перейдём к RL-обучению. В этом примере будем обучать **LoRA-адаптеры**, а не всю модель.

In [ ]:
import peft
peft_config = peft.LoraConfig(
    task_type=peft.TaskType.CAUSAL_LM, r=32, lora_alpha=32, lora_dropout=0.0, inference_mode=False
)

# повторно загрузи модель как AutoModelForCausalLMWithValueHead с дополнительной головой для PPO
main_tokenizer = transformers.AutoTokenizer.from_pretrained("lvwerra/gpt2-imdb")
main_tokenizer.pad_token = main_tokenizer.eos_token

main_model = trl.AutoModelForCausalLMWithValueHead.from_pretrained("lvwerra/gpt2-imdb", device_map=device)
main_model = peft.get_peft_model(main_model, peft_config, adapter_name='default')
main_model.print_trainable_parameters()

Как и прежде, TRL предоставляет специальный Trainer для оптимизации PPO-специфической псевдофункции потерь. Подробнее — [документация PPOTrainer](https://huggingface.co/docs/trl/main/en/ppo_trainer).

In [ ]:
training_args = trl.PPOConfig(
    model_name=main_model.config._name_or_path,
    gradient_accumulation_steps=1,
    learning_rate=1.41e-5,
    batch_size=64,
    ppo_epochs=4,                 # PPO performs this many updates per training batch
)

ppo_trainer = trl.PPOTrainer(
    training_args, model=main_model.model, tokenizer=main_tokenizer,
    dataset=imdb_for_rlhf, data_collator=lambda data: dict((key, [d[key] for d in data]) for key in data[0])
)  # note: we pass main_model.model because PPOTrainer checks for one of several supported model types ...
# ... main_model.model is a model with adapters, which is supported. main_model itself is a wrapper that is not supported

In [ ]:
from tqdm.auto import tqdm
max_steps = 50   # can be insufficient for some tasks - watch your learning curves
generation_kwargs = dict(
    min_length=-1, max_new_tokens=128, do_sample=True, top_k=0, top_p=1.0, pad_token_id=main_tokenizer.eos_token_id)
#                                  ^-- task-specific parameter!
with tqdm(enumerate(ppo_trainer.dataloader), total=max_steps) as progressbar:
  # ppo_trainer.dataloader — обычный загрузчик запросов, здесь нет особой магии RL :)
  for epoch, batch in progressbar:
    if epoch >= max_steps:
        break

    # Этап Rollout: сгенерируй продолжения для батча запросов через main_model
    response_tensors = ppo_trainer.generate(batch['input_ids'], **generation_kwargs)
    # ^-- list of tensors of token ids from main model tokenizer

    # декодируй ответы в строки, так как reward model использует другой токенизатор
    batch["response"] = [main_tokenizer.decode(response.squeeze()) for response in response_tensors]
    # response_tensors уже содержат токены запроса, поэтому добавлять их отдельно не нужно.
    # Для других задач это может быть неверно — проверь вручную batch["response"] и batch["query"]


    # Этап Evaluation
    rewards = compute_reward(batch['response'])

    # Этап Update
    stats = ppo_trainer.step(batch['input_ids'], response_tensors, list(rewards.split(1)))
    stats['rewards/mean'] = rewards.mean().item()

    print("-" * 30, 'STEP', epoch, '-' * 30)
    print(f'rewards/mean:\t{stats["rewards/mean"]:.9f}\t<---- average reward over this batch (higher=better, noisy)')
    print(f'ppo/returns/mean:\t{stats["ppo/returns/mean"]:.9f}\t<---- model-estimated average discounted reward')
    print(f'objective/kl:\t{stats["objective/kl"]:.9f}\t<---- how far we are from the original model (regularizer)')
    print()

    ppo_trainer.log_stats(stats, batch, list(rewards.split(1)))

# [Необязательно] Сложное бонусное задание: RL-дообучение на собственной задаче

Используй конвейер RLHF, чтобы обучить модель максимизировать выбранное тобой вознаграждение. Возможные варианты:

**A. Управление токсичностью:** научи модель генерировать менее (или более) токсичные тексты. Для этого можно взять данные из [Jigsaw Toxic Comments](https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge), [lmsys/toxic-chat](https://huggingface.co/datasets/lmsys/toxic-chat) или других источников; также можно использовать оценки токсичности из [oasst1](https://huggingface.co/datasets/OpenAssistant/oasst1).

**B. Настоящая обратная связь людей:** возьми датасет с попарными человеческими предпочтениями: [Anthropic hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf), [OpenAssistant](https://huggingface.co/datasets/OpenAssistant/oasst1) или подходящий другой. При желании можно даже экспериментировать с минимизацией предпочтений, если модель не деградирует до бессвязного текста.

**C. Управляемая генерация:** вместо обученной по людям reward model можно задать явную функцию вознаграждения: длину текста (увеличивать или уменьшать) либо число вхождений определённых слов, например `sorry` или `apologize`. Если выбираешь отдельные слова, убедись, что модель хотя бы иногда их генерирует.

**Можно предложить и собственную задачу**, но если она существенно отличается от рекомендованных, сложность, вычислительные затраты и время настройки могут вырасти на порядки. При сомнениях проконсультируйся с преподавателями; если они недоступны, лучше выбрать предложенный вариант.

#### Общие рекомендации

- На этапе PPO reward model должна быть в режиме `eval` (dropout отключён).
- `max_length` и `max_new_tokens` должны быть достаточно велики для большинства примеров датасета.
- Если есть сомнения — вручную посмотри несколько записей и запусков модели.

Рекомендуется проверять каждый этап отдельно:

1. **После создания парного датасета** декодируй несколько примеров из собственного `Dataset`. Убедись, что человеку понятно, почему один вариант принят, а другой отвергнут. Это помогает обнаружить ошибки токенизации и обрезки.
2. **После обучения reward model** измерь её самостоятельное качество. Если она плохо оценивает пары, последующие этапы RLHF тоже не будут работать.
3. **После RL-дообучения основной модели** сгенерируй тексты и оцени их качество. Если результат плох, проверь балл reward model. Если она считает плохой ответ хорошим — проблема в оценщике; если нет — возможно, нужен более удачный или длительный PPO.

**Во время обучения полезно периодически выводить примеры генерации.**

**Если застрял — упрости задачу.** Например, когда reward model на `hh-rlhf` не обучается, попробуй использовать 10% самых коротких последовательностей — они обычно легче.

## Этапы бонусного задания

Какую бы задачу ты ни выбрал, решение должно включать несколько **отдельно оцениваемых частей**.

#### Этап 1. Reward model

Подготовь датасет для обучения reward model на выбранной задаче. Обучи модель вознаграждения и измерь, насколько хорошо она предсказывает предпочтения на **отложенной тестовой выборке**.

Раздел с оценкой reward model в ноутбуке должен быть чётко выделен и понятен.

Не называй датасет `IMDB`, если это **не** реальные отзывы IMDB.

**Не всем задачам нужна reward model для PPO.** Например, ради длины предложения нет смысла учить отдельный оценщик. Токсичность можно измерять готовым классификатором.

Но **для получения баллов**, даже если выбранной задаче reward model не нужна, обучи **отдельную, не связанную с ней модель** на [hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf) как для варианта B. На втором этапе она использоваться не будет.

#### Этап 2. RL fine-tuning

Когда reward model обучена или есть иной способ считать вознаграждение, начинай максимизировать его через PPO. При желании можно заменить PPO другим RL-алгоритмом или схемой unlikelihood learning.

Выбери языковую модель. Подойдёт любая, но она **должна уметь генерировать данные нужного формата**.

Например, [Mistral-7B](https://huggingface.co/mistralai/Mistral-7B-v0.1) — модель общего назначения, и для ответа как чат-ассистент ей может потребоваться дополнительный промптинг. Поэтому **не используй `"lvwerra/gpt2-imdb"` для задач, отличных от генерации отзывов о фильмах**.

Есть два уровня сложности:

- **Простой:** используй [gpt2-large](https://huggingface.co/gpt2-large) или [opt-1.3b](https://huggingface.co/facebook/opt-1.3b) с минимальными изменениями кода.
- **Сложный:** используй модель размера около 7B с `load_in_4bit` и LoRA, аналогично прошлой неделе. Возможные варианты: [LLaMA-7B](https://huggingface.co/Enoch/llama-7b-hf), [Falcon-7B](https://huggingface.co/tiiuae/falcon-7b), [Mistral-7B](https://huggingface.co/mistralai/Mistral-7B-v0.1) для универсальных задач, [guanaco-7b](https://huggingface.co/timdettmers/guanaco-7b) или [vicuna-7b](https://huggingface.co/lmsys/vicuna-7b-v1.5) для чата. Есть и [другие модели](https://huggingface.co/spaces/HuggingFaceH4/open_llm_leaderboard).

В сложном варианте придётся изменить параметры обучения для 4-битного fine-tuning; эксперименты займут больше времени, зато результаты потенциально будут качественнее.

**Высокого reward недостаточно!** RL-алгоритмы умеют [обманывать функцию вознаграждения](https://openai.com/research/faulty-reward-functions). Поэтому нужна **дополнительная проверка результата**.

Для максимального балла приведи **как минимум 20 попарных примеров** генераций до и после дообучения, сопоставленных друг с другом, и краткий вывод.

Альтернатива — **пять примеров** плюс отдельная внешняя метрика на большом числе текстов.

Например:
- Для варианта A можно применить **другой** предобученный классификатор токсичности.
- Для человеческих предпочтений можно привлечь [людей](https://toloka.ai/) или [судью GPT/Claude](https://arxiv.org/pdf/2304.03277.pdf).
- Для варианта C, где награда — длина текста или подобная простая характеристика, достаточно сравнить **гистограммы reward**, например среднюю длину генерации.